# Pulse ODMR — Board-Level Test Suite
---
**Platform**: ZYNQ-7020 (XC7Z020) + PYNQ v2.7  
**Overlay**: `pulse_odmr.bit` + `pulse_odmr.hwh` in `pynq/overlay/`  
**IP Name**: `pulse_odmr_ip_0` (VLNV: `odmr.lab:user:pulse_odmr_ip:1.0`)

Run each cell in order. PASS/FAIL reported per test.  
Connect oscilloscope to J4 Pin14-17 to observe pulse outputs.

## Cell 0: Load Overlay + Import Modules

In [ ]:
import os, sys, time

SCRIPT_DIR = os.path.dirname(os.path.abspath(""))
if SCRIPT_DIR and SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

from overlay_loader import load_overlay, get_ip, IP_NAME, OVERLAY_BIT
from pulse_odmr_ip import (
    PulseOdmrIP, REG_ADDR,
    CTRL_ENABLE, CTRL_PULSE_EN, CTRL_ADC_EN, CTRL_DMA_EN, CTRL_SEQ_START,
    STAT_READY, STAT_ADC_OVF, STAT_FIFO_EMPTY, STAT_FIFO_FULL, STAT_SEQ_DONE,
    SEQ_NV_ESR, SEQ_RABI, SEQ_T1, SEQ_T2, SEQ_XY8_N, SEQ_CORR_SPEC,
)
from pulse_sequences import PulseSequences, freq_to_field
from ad_collect import ADCollector

ol = load_overlay()
ip_raw = get_ip(ol)
ip = PulseOdmrIP(ip_raw)

pass_count = 0
fail_count = 0

def T(name, condition, detail=""):
    global pass_count, fail_count
    if condition:
        pass_count += 1
        print("  PASS: %s" % name)
    else:
        fail_count += 1
        s = "  FAIL: %s" % name
        if detail:
            s += " — %s" % detail
        print(s)

print("Overlay loaded : %s" % ol.is_loaded())
print("Available IPs  : %s" % list(ol.ip_dict.keys()))
print("IP name         : %s" % IP_NAME)

## Cell 1: Register Readback (Defaults)

In [ ]:
r = ip._read(REG_ADDR["CTRL_REG"])
T("CTRL_REG default = 0x00000000", r == 0x00000000, "got 0x%08X" % r)

r = ip._read(REG_ADDR["STAT_REG"])
T("STAT_REG readable", r is not None, "got 0x%08X" % r)

r = ip._read(REG_ADDR["PULSE0_CFG"])
T("PULSE0_CFG default = 0x00001388 (5000ns)", r == 0x00001388, "got 0x%08X" % r)

r = ip._read(REG_ADDR["PULSE1_CFG"])
T("PULSE1_CFG default = 0x00140028 (40ns/20ns)", r == 0x00140028, "got 0x%08X" % r)

r = ip._read(REG_ADDR["PULSE_SEQ"])
T("PULSE_SEQ default = 0x0000000F", r == 0x0000000F, "got 0x%08X" % r)

r = ip._read(REG_ADDR["MW_FREQ_REG"])
T("MW_FREQ_REG default = 0x002BCA00 (2.87GHz)", r == 0x002BCA00, "got 0x%08X" % r)

r = ip._read(REG_ADDR["SEQ_PARAM0"])
T("SEQ_PARAM0 default = 1 (N=1)", r == 0x00000001, "got 0x%08X" % r)

r = ip._read(REG_ADDR["SEQ_PARAM1"])
T("SEQ_PARAM1 default = 0x0000017A (tau=378ns)", r == 0x0000017A, "got 0x%08X" % r)

## Cell 2: Register Write / Readback

In [ ]:
TEST = 0xDEADBEEF
ip._write(REG_ADDR["PULSE0_CFG"], TEST)
r = ip._read(REG_ADDR["PULSE0_CFG"])
T("PULSE0_CFG write 0xDEADBEEF readback", r == TEST, "0x%08X" % r)

ip._write(REG_ADDR["PULSE1_CFG"], 0x00000258)
r = ip._read(REG_ADDR["PULSE1_CFG"])
T("PULSE1_CFG write 600ns width", r == 0x00000258, "0x%08X" % r)

ip._write(REG_ADDR["MW_FREQ_REG"], 2870000)
r = ip._read(REG_ADDR["MW_FREQ_REG"])
T("MW_FREQ write 2.87GHz", r == 2870000, "got %d" % r)

ip._write(REG_ADDR["SEQ_PARAM0"], 4)
r = ip._read(REG_ADDR["SEQ_PARAM0"])
T("SEQ_PARAM0 write N=4", r == 4, "got %d" % r)

ip._write(REG_ADDR["PULSE0_CFG"], 0x00001388)
ip._write(REG_ADDR["PULSE1_CFG"], 0x00140028)
ip._write(REG_ADDR["MW_FREQ_REG"], 0x002BCA00)
ip._write(REG_ADDR["SEQ_PARAM0"], 1)
print("Restored defaults")

## Cell 3: High-Level API (set_pulse, set_mw_freq, etc.)

In [ ]:
ip.set_pulse0(5000, 0)
r = ip._read(REG_ADDR["PULSE0_CFG"])
T("set_pulse0(5000ns,0) -> 0x00001388", r == 0x00001388, "0x%08X" % r)

ip.set_pulse1(40, 20)
r = ip._read(REG_ADDR["PULSE1_CFG"])
T("set_pulse1(40ns,20ns) -> 0x00140028", r == 0x00140028, "0x%08X" % r)

ip.set_pulse_mask(0xF)
r = ip._read(REG_ADDR["PULSE_SEQ"])
T("set_pulse_mask(0xF)", r == 0x0000000F, "0x%08X" % r)

ip.set_mw_freq(2870000)
r = ip._read(REG_ADDR["MW_FREQ_REG"])
T("set_mw_freq(2870000)", r == 2870000, "%d" % r)

ip.set_mmcm_cfg(16, 1, 25, 80)
r = ip._read(REG_ADDR["MMCM_CFG_REG"])
T("set_mmcm_cfg(16,1,25,80) -> 0x50190110", r == 0x50190110, "0x%08X" % r)

ip.set_adc_sample_count(1024)
r = ip._read(REG_ADDR["ADC_CFG_REG"])
T("set_adc_sample_count(1024)", (r & 0xFFFF) == 1024, "0x%08X" % r)

## Cell 4: CTRL_REG Bit Manipulation

In [ ]:
ip._write(REG_ADDR["CTRL_REG"], 0x00000000)
r = ip._read(REG_ADDR["CTRL_REG"])
T("CTRL_REG clear -> 0x00", r == 0, "0x%08X" % r)

ip.set_ctrl(enable=True)
r = ip._read(REG_ADDR["CTRL_REG"])
T("set_ctrl(enable=1) -> bit0=1", r == 1, "0x%08X" % r)

ip.set_ctrl(enable=True, pulse_en=True)
r = ip._read(REG_ADDR["CTRL_REG"])
T("set_ctrl(pulse_en=1) -> bit[0:1]=11", r == 3, "0x%08X" % r)

ip.set_ctrl(enable=True, pulse_en=True, seq_start=True)
r = ip._read(REG_ADDR["CTRL_REG"])
T("set_ctrl(seq_start=1) -> bit4=1 appeared", (r & 0x10) != 0, "0x%08X" % r)

ip.set_ctrl(enable=True, adc_en=True, dma_en=True)
r = ip._read(REG_ADDR["CTRL_REG"])
T("set_ctrl(adc_en=1,dma_en=1) -> bit[2:3]=11", (r & 0x0C) == 0x0C, "0x%08X" % r)

ip._write(REG_ADDR["CTRL_REG"], 0x00000000)
print("CTRL_REG cleared")

## Cell 5: Status Register + Sequence Select

In [ ]:
r = ip.get_status()
print("STAT_REG = 0x%08X" % r)
T("STAT_REG readable", isinstance(r, int))
T("No ADC overflow at boot", not ip.is_adc_overflow())
T("is_ready() returns bool", isinstance(ip.is_ready(), bool))

for name, val in [("NV_ESR", 0), ("RABI", 1), ("T1", 2), ("T2", 3), ("XY8-N", 4), ("CORR", 5)]:
    ip.select_sequence(val)
    r = ip._read(REG_ADDR["SEQ_SEL"])
    T("SEQ_SEL=%s (%d)" % (name, val), r == val, "got %d" % r)

## Cell 6: Pulse Output Test (CHECK OSCILLOSCOPE)

Connect oscilloscope probes to J4 expansion port:
- **J4 Pin14** → pulse_0 (AOM control)
- **J4 Pin15** → pulse_1 (microwave switch)
- **J4 Pin16** → pulse_2 (peripheral trigger)
- **J4 Pin17** → pulse_3 (global sync)

Expected: pulse_0 = 5us wide pulse, pulse_1 = 40ns pulse with 20ns delay

In [ ]:
ip.set_pulse0(5000, 0)
ip.set_pulse1(40, 20)
ip.set_pulse2(5000, 0)
ip.set_pulse3(5000, 0)
ip.set_pulse_mask(0xF)
ip.select_sequence(0)

print("Triggering pulse sequence (CTRL=enable + pulse_en + seq_start)...")
ip.set_ctrl(enable=True, pulse_en=True, seq_start=True)
print("CTRL_REG = 0x%08X" % ip._read(REG_ADDR["CTRL_REG"]))
print("Pulse sequence triggered! Check oscilloscope on J4 Pin14-17.")

## Cell 7: Magnetic Field Conversion Utilities

In [ ]:
field = freq_to_field(2870000, 2870000)
T("freq_to_field(2.87GHz) -> 0G (on resonance)", abs(field) < 0.01, "%.2f G" % field)

field = freq_to_field(2800000, 2870000)
expected = 70000.0 / 2.8
T("freq_to_field(2.80GHz) -> ~25000G", abs(field - expected) < 1,
  "%.1f G (expected ~%.0f)" % (field, expected))

print("NV zero-field splitting D = 2.87 GHz")
print("Gyromagnetic ratio gamma = 2.8 MHz/G")
print("B_field = |D - f_NV| / gamma")

## Cell 8: Sequence API Sanity

Verify PulseSequences wrapper instantiates correctly.

In [ ]:
seq = PulseSequences(ip)
T("PulseSequences created", seq is not None)
T("seq.ip is correct", seq.ip is ip)

print("\nSequence defaults:")
print("  NV zero-field  : %d kHz (%.3f GHz)" % (2870000, 2870000/1e6))
print("  Gyromagnetic   : %.1f MHz/G" % 2.8)
print("  Default pi pulse: %d ns" % 40)
print("  Default AOM     : %d ns" % 5000)
print("  Default tau     : %d ns" % 378)

## Cell 9: AD Collector Init

In [ ]:
adc = ADCollector(ip, sample_points=1024)
T("ADCollector created", adc.sample_points == 1024)

r = ip._read(REG_ADDR["ADC_CFG_REG"])
T("ADC_CFG sample count set to 1024", (r & 0xFFFF) == 1024, "0x%08X" % r)

r = ip._read(REG_ADDR["DMA_CFG_REG"])
T("DMA_CFG length set to 1024", (r & 0xFFFF) == 1024, "0x%08X" % r)

## Cell 10: Full Register Dump

In [ ]:
from pl_reg_driver import PLRegDriver

reg_drv = PLRegDriver()
print("DEFAULT values from register_map.csv (expected):")
print("-" * 55)
reg_drv.dump_all()

errors = reg_drv.verify_defaults()
if errors:
    print("\nREGS DIFFERING FROM DEFAULTS:")
    for e in errors:
        print("  %s" % e)
else:
    print("\nAll registers match hardware defaults.")

## Cell 11: Cleanup + Summary

In [ ]:
ip.system_disable()
T("System disable after tests", ip._read(REG_ADDR["CTRL_REG"]) == 0)

print()
print("=" * 60)
print(" RESULTS: %d PASS, %d FAIL, %d TOTAL" % (pass_count, fail_count, pass_count + fail_count))
print("=" * 60)

if fail_count == 0:
    print("\nAll tests passed! The pulse_odmr IP is functional.")
    print("Next: connect oscilloscope to J4 Pin14-17 and run Cell 6.")
else:
    print("\n%d test(s) failed. Check the FAIL lines above." % fail_count)